# UC-UW-3 — Read-side UI bindings: Search, Features, MVT Tiles

**Goal:** Drive the three read endpoints the alternative UI needs:
1. **Search** — `POST /search` (ES-primary; cursor pagination via `token`).
2. **List features** — `GET /features/.../items` (PG-backed; CQL2 + bbox + offset).
3. **MVT tiles** — `GET /tiles/catalogs/{cat}/tiles/{tms}/{z}/{x}/{y}.mvt`.

**Prereq:** notebook 02 ran successfully (collection has at least one item ingested).


In [ ]:
import json
import os
import time
import uuid

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass
TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)
RUN_ID = os.environ.get("RUN_ID", uuid.uuid4().hex[:8])
CATALOG_ID = os.environ.get("CATALOG_ID", f"uw_{RUN_ID}")
COLLECTION_ID = os.environ.get("COLLECTION_ID", f"col_{RUN_ID}")

# REMOTE_ONLY heuristic: pub/sub-driven flows do not fire against localhost.
IS_LOCAL = "localhost" in BASE_URL or "127.0.0.1" in BASE_URL
PUBSUB_ENABLED = not IS_LOCAL

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"

client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=120.0)

print(f"BASE_URL      : {BASE_URL}")
print(f"CATALOG_ID    : {CATALOG_ID}")
print(f"COLLECTION_ID : {COLLECTION_ID}")
print(f"RUN_ID        : {RUN_ID}")
print(f"AUTH          : {'token set' if TOKEN else 'anonymous'}")
print(f"PUBSUB        : {'enabled (remote)' if PUBSUB_ENABLED else 'disabled (local)'}")


## 1 — STAC item search (ES-primary, cursor pagination)

In [ ]:
# 1) Search — STAC item search (ES-primary)
search_body = {
    "collections": [COLLECTION_ID],
    "limit": 5,
}
r = client.post("/search", content=json.dumps(search_body))
print(f"POST /search: {r.status_code}")
assert r.status_code == 200, f"search: {r.text[:300]}"
body = r.json()
features = body.get("features", [])
print(f"features returned: {len(features)}")
print(f"numberMatched: {body.get('numberMatched')}  numberReturned: {body.get('numberReturned')}")

# Cursor pagination — the platform returns a `token` for the next page when
# numberReturned == limit and more results exist.
token = body.get("token") or next(
    (l.get("body", {}).get("token") for l in body.get("links", []) if l.get("rel") == "next"),
    None,
)
print(f"next page token: {token!r}")
if token:
    body2 = client.post("/search", content=json.dumps({**search_body, "token": token})).json()
    print(f"page 2 returned: {len(body2.get('features', []))} features")


## 2 — Geoid lookup

In [ ]:
# 2) Geoid lookup — used by selection panels in the UI
if features:
    sample_geoid = features[0].get("properties", {}).get("geoid") or features[0].get("id")
    r = client.post(
        f"/search/catalogs/{CATALOG_ID}/items-search",
        content=json.dumps({"geoid": sample_geoid}),
    )
    print(f"geoid lookup: {r.status_code}")
    if r.status_code == 200:
        body = r.json()
        results = body.get("results") or body.get("features") or body
        print(json.dumps(results, indent=2)[:600])
else:
    print("(no features in collection — skip geoid lookup)")

## 3 — OGC Features list (CQL2, bbox, sort)

In [ ]:
# 3) List features — OGC API Features (PG-backed)
features_url = f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items"
r = client.get(features_url, params={"limit": 5})
print(f"GET features: {r.status_code}")
assert r.status_code == 200, f"features: {r.text[:300]}"
fc = r.json()
print(f"numberMatched: {fc.get('numberMatched')}  numberReturned: {fc.get('numberReturned')}")

# CQL2 filter — only Rome
r = client.get(
    features_url,
    params={
        "limit": 5,
        "filter": "name LIKE 'Rome%'",
        "filter_lang": "cql2-text",
    },
)
print(f"GET features (CQL2 filter): {r.status_code}")
if r.status_code == 200:
    fc = r.json()
    print(f"  filtered count: {fc.get('numberReturned')}")
    for feat in fc.get("features", [])[:3]:
        print(f"    {feat.get('id')}: {feat.get('properties', {}).get('name')}")

# bbox + sortby
r = client.get(features_url, params={"limit": 10, "bbox": "-10,40,15,52", "sortby": "+name"})
print(f"GET features (bbox + sort): {r.status_code} count={r.json().get('numberReturned') if r.status_code == 200 else '-'}")


## 4 — MVT vector tile

In [ ]:
# 4) MVT tile fetch — pick a tile that covers Rome at z=4
mvt_url = f"/tiles/catalogs/{CATALOG_ID}/tiles/WebMercatorQuad/4/8/5.mvt"
r = client.get(mvt_url, params={"collections": COLLECTION_ID})
print(f"GET tile {mvt_url}: {r.status_code}, {len(r.content)} bytes")
print(f"content-type: {r.headers.get('content-type')}")
# Decode optionally — only if mapbox_vector_tile is installed in the kernel.
try:
    import mapbox_vector_tile
    if r.status_code == 200 and r.content:
        decoded = mapbox_vector_tile.decode(r.content)
        for layer_name, layer in decoded.items():
            print(f"  layer {layer_name!r}: {len(layer.get('features', []))} features")
except Exception as e:
    print(f"  (mapbox_vector_tile not installed in kernel — skipped decode: {type(e).__name__})")

client.close()
